# LoRA finetuning a released AstroPT checkpoint

Loads a real pretrained checkpoint from
[`Smith42/astroPT`](https://huggingface.co/Smith42/astroPT) (Smith et al.,
2024, https://arxiv.org/abs/2405.14930) and LoRA-finetunes it (Hu et al.,
2021, https://arxiv.org/abs/2106.09685) for galaxy morphology classification
on [`UniverseTBD/mmu_gz10`](https://huggingface.co/datasets/UniverseTBD/mmu_gz10)
(Galaxy10 DECals, 17,736 galaxies, 10 classes), following the reference
model's own `scripts/finetune.py` recipe: freeze the pretrained weights,
train only injected low-rank adapters and a new task head.

This uses the 89M-parameter checkpoint; other sizes (1M to 2.1B) are listed
at the same repo under `models/fully_trained/`.

**Checkpoint compatibility.** The released checkpoints predate this
library's simplified, single-modality reimplementation, so only part of the
checkpoint loads — see `load_pretrained_backbone` below for what transfers
and why.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [1]:
!pip install -q datasets torchvision scikit-learn huggingface_hub

## Imports

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from utils import gz10
from utils.astropt import load_pretrained_astropt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the dataset and split 70/10/20

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 64
CLASS_NAMES = gz10.CLASS_NAMES
NUM_CLASSES = gz10.NUM_CLASSES

data, labels, train_idx, val_idx, test_idx = gz10.load_split_702010()

# no pixel Normalize here: AstroPT.patchify normalizes each patch to zero
# mean/unit variance itself, matching the reference model's pretraining input
train_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomRotation(90),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
    ]
)

train_dataset = gz10.GZ10Dataset(data, train_idx, train_transform)
val_dataset = gz10.GZ10Dataset(data, val_idx, eval_transform)
test_dataset = gz10.GZ10Dataset(data, test_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)

len(train_dataset), len(val_dataset), len(test_dataset)

Resolving data files:   0%|          | 0/921 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/766 [00:00<?, ?it/s]

(12415, 1773, 3548)

## Download and load the pretrained backbone

`Smith42/astroPT` checkpoints predate this library's multimodal-free
reimplementation: they store the causal transformer under `transformer.h.*`
(compiled with `torch.compile`, hence the `_orig_mod.` prefix to strip) with
a patch encoder/decoder (`transformer.wte.*` / `lm_head.*`) too deep to line
up with `AstroPT`'s single-layer patch projection. `load_pretrained_astropt`
(`utils/astropt.py`, shared across the AstroPT notebooks) builds a model
matching the checkpoint's dimensions and transfers everything that lines up
exactly — attention, MLP, layer norms, and the learned position embeddings
(sliced to how many patches this image size needs) — leaving the encoder,
decoder, LoRA adapters, and classification head freshly initialized, to be
learned during finetuning.

In [ ]:
LORA_R = 8

model = load_pretrained_astropt(IMG_SIZE, device, num_classes=NUM_CLASSES, lora_r=LORA_R)
model.mark_only_lora_as_trainable()

sum(p.numel() for p in model.parameters() if p.requires_grad)

## Finetune

In [5]:
FINETUNE_EPOCHS = 15
FINETUNE_LR = 1e-4  # matches the reference model's finetune.py recipe

# inverse-frequency class weights from the train split, following the
# Linformer notebook's approach to counter GZ10's class imbalance
criterion = nn.CrossEntropyLoss(weight=gz10.class_weights(labels, train_idx).to(device))
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=FINETUNE_LR
)

for epoch in range(1, FINETUNE_EPOCHS + 1):
    train_loss, train_acc = gz10.run_classification_epoch(
        model, train_loader, criterion, device, train=True, optimizer=optimizer
    )
    val_loss, val_acc = gz10.run_classification_epoch(model, val_loader, criterion, device, train=False)
    print(
        f"epoch {epoch:02d} train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
    )

epoch 01 train_loss=1.8336 train_acc=0.358 val_loss=1.5928 val_acc=0.432


epoch 02 train_loss=1.4057 train_acc=0.509 val_loss=1.3738 val_acc=0.505


epoch 03 train_loss=1.3014 train_acc=0.545 val_loss=1.3297 val_acc=0.528


epoch 04 train_loss=1.2406 train_acc=0.566 val_loss=1.2480 val_acc=0.564


epoch 05 train_loss=1.2096 train_acc=0.582 val_loss=1.2495 val_acc=0.568


...

epoch 11 train_loss=1.0091 train_acc=0.654 val_loss=1.0241 val_acc=0.631


epoch 12 train_loss=0.9911 train_acc=0.655 val_loss=1.0373 val_acc=0.631


epoch 13 train_loss=0.9802 train_acc=0.660 val_loss=1.0487 val_acc=0.638


epoch 14 train_loss=0.9714 train_acc=0.665 val_loss=1.0378 val_acc=0.641


epoch 15 train_loss=0.9613 train_acc=0.665 val_loss=0.9612 val_acc=0.663


## Evaluate on the held-out test split

In [6]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels_batch in test_loader:
        logits = model(images.to(device))
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_true.extend(labels_batch.tolist())

test_acc = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"test_acc={test_acc:.3f} test_f1_macro={test_f1_macro:.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

test_acc=0.664 test_f1_macro=0.628

                         precision    recall  f1-score   support

              disturbed       0.38      0.36      0.37       216
                merging       0.59      0.60      0.60       371
           round_smooth       0.83      0.88      0.85       529
in_between_round_smooth       0.72      0.86      0.79       405
    cigar_shaped_smooth       0.28      0.81      0.42        67
          barred_spiral       0.65      0.67      0.66       409
  unbarred_tight_spiral       0.58      0.67      0.62       366
  unbarred_loose_spiral       0.65      0.30      0.41       525
       edge_on_no_bulge       0.75      0.90      0.82       285
     edge_on_with_bulge       0.86      0.67      0.75       375

               accuracy                           0.66      3548
              macro avg       0.63      0.67      0.63      3548
           weighted avg       0.68      0.66      0.66      3548

